# Local LLM -> JSON -> validator repair loop

Этот ноутбук добавляет экспериментальный ML-слой поверх существующего генератора макросов.

Схема:

```text
текст пользователя -> локальная LLM -> JSON -> resolve_placements -> validate_plan -> .m3m
                                      ^ если ошибка, отправляем ее модели на исправление
```

Важно: модель не получает узких команд вроде `create_birdhouse`. Она должна собирать детали из базовых команд и универсального `attach`/`placement`.

## 1. Установка

Запусти эту ячейку только если в окружении еще нет `llama-cpp-python`.

Для T4 обычно удобно начинать с GGUF-файла `Qwen2.5-Coder-14B-Instruct-Q4_K_M.gguf` или, если хочется быстрее, с `Qwen2.5-Coder-7B-Instruct-Q5_K_M.gguf`.

In [ ]:
# Раскомментируй при первом запуске в новом окружении.
# %pip install -U llama-cpp-python


## 2. Импорты и пути

In [ ]:
from __future__ import annotations

import json
import os
import re
import sys
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

from macro_generator import write_macro
from placement_resolver import resolve_placements
from validator import ValidationError, validate_plan

WORK_DIR = PROJECT_ROOT / "ml_json_generator"
GENERATED_DIR = WORK_DIR / "generated"
LOG_DIR = WORK_DIR / "logs"
GENERATED_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = LOG_DIR / "json_repair_log.jsonl"
SFT_PATH = LOG_DIR / "sft_dataset.jsonl"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LOG_PATH:", LOG_PATH)

## 3. Модель

`MODEL_PATH` должен указывать на локальный `.gguf` файл. Можно задать переменную окружения `CAD_LLM_MODEL_PATH`, чтобы не менять ноутбук.

In [ ]:
MODEL_PATH = Path(os.getenv("CAD_LLM_MODEL_PATH", r"models/Qwen2.5-Coder-14B-Instruct-Q4_K_M.gguf"))
N_CTX = 8192
N_GPU_LAYERS = -1
TEMPERATURE = 0.15
MAX_TOKENS = 2500
MAX_REPAIR_ATTEMPTS = 3

print("MODEL_PATH:", MODEL_PATH)

In [ ]:
class LocalGGUFModel:
    def __init__(self, model_path: Path, n_ctx: int = 8192, n_gpu_layers: int = -1):
        try:
            from llama_cpp import Llama
        except ImportError as error:
            raise RuntimeError("Install llama-cpp-python first") from error

        if not model_path.exists():
            raise FileNotFoundError(f"GGUF model not found: {model_path}")

        self.llm = Llama(
            model_path=str(model_path),
            n_ctx=n_ctx,
            n_gpu_layers=n_gpu_layers,
            verbose=False,
        )

    def chat(self, messages: list[dict], temperature: float = 0.15, max_tokens: int = 2500) -> str:
        result = self.llm.create_chat_completion(
            messages=messages,
            temperature=temperature,
            max_tokens=max_tokens,
        )
        return result["choices"][0]["message"]["content"]


# Запускай эту ячейку, когда модель уже скачана.
# model = LocalGGUFModel(MODEL_PATH, n_ctx=N_CTX, n_gpu_layers=N_GPU_LAYERS)

## 4. Спецификация JSON-языка для модели

Это главный текст, который удерживает модель в рамках наших универсальных команд.

In [ ]:
CAD_JSON_SPEC = """
You generate JSON plans for a KOMPAS macro generator.
Return only valid JSON. Do not use Markdown. Do not explain anything.

Top-level object:
{
  "version": "0.1",
  "units": "mm",
  "part_name": "ascii_snake_case_name",
  "commands": [ ... ]
}

Allowed command types only:
- create_box
- cut_box
- create_prism
- cut_prism
- create_triangle
- cut_triangle
- create_cylinder
- cut_cylinder

Never create object-specific command types. Forbidden examples:
- create_birdhouse
- create_plate_with_holes
- create_screwdriver
- create_table

Low-level create_box / cut_box:
{
  "id": "body",
  "type": "create_box",
  "origin": [x, y, z],
  "size": [height_on_sketch, extrusion_depth, width_on_sketch],
  "plane": "YOZ",
  "extrude": "middle",
  "select_point": [x, y, z]
}

Semantic placed create_box / cut_box:
{
  "id": "body",
  "type": "create_box",
  "size": [width_y, height_z, depth_x],
  "placement": {"origin": [x, y, z]}
}

Triangle prism on top of a box:
{
  "id": "roof",
  "type": "create_triangle",
  "size": [width_y, height_z, depth_x],
  "attach": {"target": "body", "face": "top", "position": "center"}
}

Cylinder or cylindrical cut attached to front face of a box:
{
  "id": "hole",
  "type": "cut_cylinder",
  "radius": 10,
  "depth": "half",
  "attach": {
    "target": "body",
    "face": "front",
    "position": "center",
    "offset": [dy, dz]
  }
}

Extrude modes:
- normal: extrude in normal direction
- reverse: extrude in opposite direction
- middle: extrude symmetrically around sketch plane

Rules:
- Use millimeters.
- Use stable ASCII ids.
- Every command id must be unique.
- A command can attach only to an object created earlier.
- Prefer semantic placement/attach when the user describes relative positioning.
- Use low-level coordinates when the user gives exact coordinates or exact holes.
- For a protruding perch or peg on the front face, use create_cylinder with extrude="reverse".
- For an inward hole on the front face, use cut_cylinder with extrude="normal".
""".strip()

## 5. Примеры и память

Удачные JSON из проекта и из журнала можно подмешивать в промпт. Это не настоящее обучение весов, но модель начинает лучше попадать в наш стиль.

In [ ]:
SEED_EXAMPLE_PATHS = [
    PROJECT_ROOT / "examples" / "birdhouse_attach.json",
    PROJECT_ROOT / "examples" / "plate_four_holes.json",
    PROJECT_ROOT / "examples" / "primitives_demo.json",
]


def load_json_file(path: Path) -> dict | None:
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8") as file:
        return json.load(file)


def load_seed_examples() -> list[dict]:
    examples = []
    for path in SEED_EXAMPLE_PATHS:
        data = load_json_file(path)
        if data is not None:
            examples.append(data)
    return examples


def load_memory_examples(limit: int = 3) -> list[dict]:
    if not LOG_PATH.exists():
        return []

    records = []
    with LOG_PATH.open("r", encoding="utf-8") as file:
        for line in file:
            if not line.strip():
                continue
            record = json.loads(line)
            if record.get("success") and record.get("final_plan"):
                records.append(record["final_plan"])
    return records[-limit:]


def format_examples(examples: list[dict], limit: int = 3) -> str:
    if not examples:
        return "No examples yet."
    selected = examples[-limit:]
    chunks = []
    for index, example in enumerate(selected, start=1):
        chunks.append(f"Example {index}:\n" + json.dumps(example, ensure_ascii=False, indent=2))
    return "\n\n".join(chunks)

## 6. Извлечение JSON и проверка через наш пайплайн

In [ ]:
def extract_json_object(text: str) -> dict:
    cleaned = text.strip()
    fence_match = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", cleaned, flags=re.DOTALL)
    if fence_match:
        cleaned = fence_match.group(1)

    first = cleaned.find("{")
    last = cleaned.rfind("}")
    if first == -1 or last == -1 or last <= first:
        raise ValueError("Model output does not contain a JSON object")

    return json.loads(cleaned[first:last + 1])


def validate_and_resolve(plan: dict) -> tuple[dict, bool]:
    resolved_plan, was_resolved = resolve_placements(plan)
    validate_plan(resolved_plan)
    return resolved_plan, was_resolved


def save_generated_outputs(raw_plan: dict, resolved_plan: dict) -> dict:
    part_name = resolved_plan["part_name"]
    raw_json_path = GENERATED_DIR / f"{part_name}.json"
    resolved_json_path = GENERATED_DIR / f"{part_name}.resolved.json"

    raw_json_path.write_text(json.dumps(raw_plan, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    resolved_json_path.write_text(json.dumps(resolved_plan, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    macro_path = write_macro(resolved_plan, PROJECT_ROOT / "artifacts" / "macros")

    return {
        "raw_json_path": str(raw_json_path),
        "resolved_json_path": str(resolved_json_path),
        "macro_path": str(macro_path),
    }

## 7. Промпты генерации и исправления

In [ ]:
def build_system_message() -> str:
    return (
        "You are a CAD JSON planner. "
        "You output only valid JSON for the provided schema. "
        "You never invent object-specific commands."
    )


def build_generation_messages(user_request: str, part_name: str) -> list[dict]:
    examples = load_seed_examples() + load_memory_examples(limit=3)
    user_message = f"""
{CAD_JSON_SPEC}

Known good examples:
{format_examples(examples, limit=4)}

User request in Russian:
{user_request}

Use this part_name: {part_name}

Return only the JSON plan.
""".strip()
    return [
        {"role": "system", "content": build_system_message()},
        {"role": "user", "content": user_message},
    ]


def build_repair_messages(user_request: str, part_name: str, bad_output: str, error: str) -> list[dict]:
    user_message = f"""
{CAD_JSON_SPEC}

Original user request:
{user_request}

Required part_name: {part_name}

Your previous output was invalid:
{bad_output}

Validator error:
{error}

Fix the JSON. Return only one complete valid JSON object.
""".strip()
    return [
        {"role": "system", "content": build_system_message()},
        {"role": "user", "content": user_message},
    ]

## 8. Основной цикл: генерация -> ошибка -> исправление

In [ ]:
def append_log(record: dict) -> None:
    with LOG_PATH.open("a", encoding="utf-8") as file:
        file.write(json.dumps(record, ensure_ascii=False) + "\n")


def generate_plan_with_repair(
    model: LocalGGUFModel,
    user_request: str,
    part_name: str,
    max_attempts: int = MAX_REPAIR_ATTEMPTS,
) -> dict:
    attempts = []
    messages = build_generation_messages(user_request, part_name)

    for attempt_index in range(1, max_attempts + 1):
        output = model.chat(messages, temperature=TEMPERATURE, max_tokens=MAX_TOKENS)
        attempt = {"attempt": attempt_index, "raw_output": output}

        try:
            raw_plan = extract_json_object(output)
            resolved_plan, was_resolved = validate_and_resolve(raw_plan)
            paths = save_generated_outputs(raw_plan, resolved_plan)

            attempt.update({
                "ok": True,
                "was_resolved": was_resolved,
                "paths": paths,
            })
            attempts.append(attempt)

            record = {
                "created_at": datetime.now().isoformat(timespec="seconds"),
                "success": True,
                "user_request": user_request,
                "part_name": part_name,
                "attempts": attempts,
                "final_plan": raw_plan,
                "resolved_plan": resolved_plan,
                "paths": paths,
            }
            append_log(record)
            return record

        except Exception as error:
            attempt.update({"ok": False, "error": str(error)})
            attempts.append(attempt)
            messages = build_repair_messages(user_request, part_name, output, str(error))

    record = {
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "success": False,
        "user_request": user_request,
        "part_name": part_name,
        "attempts": attempts,
        "final_plan": None,
    }
    append_log(record)
    return record

## 9. Пробный запрос

После загрузки модели раскомментируй вызов ниже.

In [ ]:
USER_REQUEST = """
Сделай скворечник: прямоугольный корпус, сверху треугольная крыша, в центре передней стороны круглое отверстие до середины корпуса, под отверстием жердочка цилиндром наружу.
""".strip()

PART_NAME = "llm_birdhouse_test"

# record = generate_plan_with_repair(model, USER_REQUEST, PART_NAME)
# print(json.dumps(record, ensure_ascii=False, indent=2))

## 10. Подготовка датасета для будущего LoRA

Эта ячейка берет успешные записи из журнала и превращает их в простой SFT-датасет формата JSONL. Это уже можно будет использовать позже для настоящего дообучения.

In [ ]:
def build_sft_dataset() -> int:
    if not LOG_PATH.exists():
        return 0

    count = 0
    with LOG_PATH.open("r", encoding="utf-8") as source, SFT_PATH.open("w", encoding="utf-8") as target:
        for line in source:
            if not line.strip():
                continue
            record = json.loads(line)
            if not record.get("success") or not record.get("final_plan"):
                continue

            sample = {
                "messages": [
                    {"role": "system", "content": build_system_message()},
                    {"role": "user", "content": CAD_JSON_SPEC + "\n\nUser request:\n" + record["user_request"]},
                    {"role": "assistant", "content": json.dumps(record["final_plan"], ensure_ascii=False)},
                ]
            }
            target.write(json.dumps(sample, ensure_ascii=False) + "\n")
            count += 1

    return count


# count = build_sft_dataset()
# print(f"SFT samples written: {count} -> {SFT_PATH}")